In [27]:
import uuid, time
from pyspark.sql.functions import col, lit
from lakehouse_engine.config import get_notebook_parameters, get_root_path, GoldConfig
from lakehouse_engine.spark import get_spark_session
from lakehouse_engine.utils import setup_logging
from lakehouse_engine.io import DataFrameReader, DataFrameWriter

In [ ]:
#PROVISORIO SPARK LOCAL
import ipywidgets as widgets
from IPython.display import display

dataset_widget = widgets.Text(value='dim_sellers', description='Dataset:')

display(dataset_widget)

Text(value='dim_customers', description='Dataset:')

In [29]:
layer = "gold"

# Load Spark Session
spark = get_spark_session()
dataset = get_notebook_parameters(dataset_widget)

# Load logging
log = setup_logging(f"{dataset}_{layer}")

# Load Gold config
gold_cfg = GoldConfig(dataset, layer)
gold_cfg.load_yaml()

[2026-03-18 11:24:34,847] [WARNING] [LOCAL] Running in local mode.


In [30]:
# Create variables from yaml
source_table = gold_cfg.source_table
source_schema = gold_cfg.source_schema
source_catalog = gold_cfg.source_catalog
target_table = gold_cfg.target_table
target_schema = gold_cfg.target_schema
target_catalog = gold_cfg.target_catalog

# Create logical variables
run_id = str(uuid.uuid4())

# Create paths
root_path = get_root_path()

In [31]:
log.info(f"[PARAMS] dataset={dataset} layer={layer}")
log.info(f"[CTX] source_table={source_table} target_table={target_table}")
log.info(f"[CTX] target_schema={target_schema} run_id={run_id}")

start = time.time()
log.info(f"[GOLD][START] run_id={run_id}")

try:
    reader = DataFrameReader(spark)
    writer = DataFrameWriter(spark)
    
    # Read table from Silver
    df_gold = reader.read_table(
        catalog=source_catalog, 
        schema=source_schema, 
        table=source_table
    )
    log.info(f"[STEP 1] READ: Silver table loaded: {source_table} rows")

    select_columns = gold_cfg.select_columns
    df_gold_selected = df_gold.select(*select_columns)
    log.info(f"[STEP 2] SELECT: Filter columns from Silver table: {select_columns}")

    writer.write_delta_batch(
        df=df_gold_selected,
        target_table=f"{target_schema}.{target_table}",
        mode="overwrite"
    )
    
    duration_s = round(time.time() - start, 3)
    log.info(f"[GOLD][SUCCESS] run_id={run_id} dataset={dataset} duration_s={duration_s}s")

except Exception as e:
    duration_s = round(time.time() - start, 3)
    log.error(f"[GOLD][FAILED] run_id={run_id} duration_s={duration_s} error={repr(e)}")
    raise

[2026-03-18 11:24:34,988] [INFO] [PARAMS] dataset=dim_customers layer=gold
[2026-03-18 11:24:34,990] [INFO] [CTX] source_table=customers target_table=dim_customers
[2026-03-18 11:24:34,991] [INFO] [CTX] target_schema=gold run_id=ef0811e9-f7c8-4d65-af9a-0b512015b7ac
[2026-03-18 11:24:34,992] [INFO] [GOLD][START] run_id=ef0811e9-f7c8-4d65-af9a-0b512015b7ac
[2026-03-18 11:24:35,017] [INFO] [STEP 1] READ: Silver table loaded: customers rows
[2026-03-18 11:24:35,044] [INFO] [STEP 2] SELECT: Filter columns from Silver table: ['customer_id', 'customer_unique_id', 'customer_city', 'customer_state', 'customer_zip_code_prefix']
[2026-03-18 11:24:37,618] [INFO] [GOLD][SUCCESS] run_id=ef0811e9-f7c8-4d65-af9a-0b512015b7ac dataset=dim_customers duration_s=2.626s
